# BioNodulo on Google Colab

This notebook launches a temporary BioNodulo instance inside a Colab runtime. Files created in Colab are ephemeral unless you download or save them elsewhere.

BioNodulo is distributed under the BioNodulo Research License. Publication, commercial use, and hosted services require a separate license.

In [ ]:
%cd /content
!test -d BioNodulo || git clone -q --branch bionodulo-collab https://github.com/Classacre/BioNodulo.git
%cd /content/BioNodulo
!git fetch -q origin bionodulo-collab
!git checkout -q bionodulo-collab
!git pull -q --ff-only origin bionodulo-collab
!python -m pip install -q .

## Start BioNodulo through Cloudflare Tunnel

Run this cell and wait for a `trycloudflare.com` URL to print. Open that URL to use BioNodulo while this cell keeps running.

The tunnel URL is public to anyone who has the link. Do not use it for sensitive data. Stop the cell when you are done to stop the BioNodulo server.

In [ ]:
!wget -q -O /content/cloudflared-linux-amd64.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /content/cloudflared-linux-amd64.deb >/dev/null

import socket
import subprocess
import threading
import time
from pathlib import Path

workspace = Path('/content/bionodulo_workspace')
workspace.mkdir(exist_ok=True)

def launch_cloudflare_tunnel(port):
    while True:
        time.sleep(0.5)
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            if sock.connect_ex(('127.0.0.1', port)) == 0:
                break

    print('\nBioNodulo is ready. Launching Cloudflare Tunnel...\n')
    tunnel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    for raw_line in tunnel.stderr:
        line = raw_line.decode(errors='replace')
        if 'trycloudflare.com' in line:
            url = line[line.find('http'):].strip()
            print(f'Open BioNodulo: {url}')
            print('Keep this cell running while you use the app.')

threading.Thread(target=launch_cloudflare_tunnel, daemon=True, args=(8000,)).start()

!python main.py --host 0.0.0.0 --port 8000 --project-root /content/bionodulo_workspace